# Vapor-Eyes 02 — Targeted detection: Sentinel-2 SWIR (MBMP)

**Step 2: zoom the wide-area hotspot to 20 m.** NB01's S5P screen flagged
candidate cells at ~7 km; here we drop to **Sentinel-2** 20 m SWIR over the
strongest hotspot to look for an actual plume signal:

- **Locate** the top `s5p_hotspots` cell (NB01) and take its footprint as the AOI.
- **Stage** Sentinel-2 L2A **B11/B12 (SWIR)** COGs for that footprint (low cloud) via `StacClient`, windowed to the cell.
- **Detect** with a multi-band SWIR index — `rst_mapalgebra` computes `(B11 − B12)/(B11 + B12)`, high where B12 (2.19 µm) absorbs relative to B11 (1.61 µm): a methane proxy.
- **Grid** the index to fine H3 cells (`rst_h3_tessellate`) for a per-cell plume fraction.

**Result:** a 20 m SWIR methane-proxy raster over the flagged hotspot.

> _Note: the SWIR band-ratio is an illustrative proxy, not an operational methane retrieval._

---
_Last Modified:_ July 11, 2026

![Top S5P hotspot cell → Sentinel-2 B11/B12 SWIR COGs → (B11-B12)/(B11+B12) index → H3 plume cells](https://raw.githubusercontent.com/databrickslabs/geobrix/main/resources/images/diagrams/vapor-eyes/vapor-eyes-02.png)

In [ ]:
%run ./config_nb

In [ ]:
CLOUD_MAX = 20      # keep Sentinel-2 items below this % cloud cover
S2_H3_RES = 10      # ~65 m H3 cells suit the 20 m SWIR index

## 1. Locate the target hotspot + its footprint (from NB01)

Take the strongest `s5p_hotspots` cell by peak CH4 and use the H3 python client to
get its boundary → a small AOI bbox. `h3_boundaryasgeojson` builds the search AOI.

In [ ]:
import h3  # noqa: E402

top_cellid = spark.table("s5p_hotspots").orderBy(F.desc("ch4_max")).first()["h3_cellid"]
# h3 cellid stored as bigint -> hex string for the h3 python client
_h = format(int(top_cellid) & 0xFFFFFFFFFFFFFFFF, "015x")
_boundary = h3.cell_to_boundary(_h)  # list[(lat, lon)]
_lats = [p[0] for p in _boundary]
_lons = [p[1] for p in _boundary]
pad = 0.03  # ~3 km pad around the cell for scene context
S2_BBOX = (min(_lons) - pad, min(_lats) - pad, max(_lons) + pad, max(_lats) + pad)
print(f"... target hotspot cell {_h}; Sentinel-2 AOI bbox {S2_BBOX}")

aoi = spark.createDataFrame(
    [(f'{{"type":"Polygon","coordinates":[[[{S2_BBOX[0]},{S2_BBOX[1]}],'
      f'[{S2_BBOX[2]},{S2_BBOX[1]}],[{S2_BBOX[2]},{S2_BBOX[3]}],'
      f'[{S2_BBOX[0]},{S2_BBOX[3]}],[{S2_BBOX[0]},{S2_BBOX[1]}]]]}}',)],
    ["geojson"],
)

## 2. Stage Sentinel-2 B11/B12 SWIR COGs (windowed, low cloud)

`StacClient.search` queries `sentinel-2-l2a` items intersecting the AOI; we keep the
**B11/B12** SWIR assets from the least-cloudy item and `StacClient.download` fetches
them **windowed to the AOI bbox** (rasterio COG windowing) into `s2_swir_assets`.

In [ ]:
if FORCE_REBUILD or not spark.catalog.tableExists("s2_swir_assets"):
    found = stac_client.search(
        aoi, geojson_col="geojson", collections=["sentinel-2-l2a"], datetime=DATE_WINDOW
    )
    cloud = F.col("item_properties")["eo:cloud_cover"].cast("double")
    best_item = (
        found.withColumn("cloud", cloud)
        .filter(F.col("cloud") <= CLOUD_MAX)
        .orderBy("cloud")
        .first()["item_id"]
    )
    bands = found.filter(
        (F.col("item_id") == best_item) & F.col("asset_name").isin("B11", "B12")
    ).select("item_id", "asset_name", "href")
    s2_dl = stac_client.download(bands, S2_DIR, bbox=list(S2_BBOX), bbox_crs="EPSG:4326")
    finalize_delta(
        s2_dl.withColumnRenamed("out_file_path", "band_path"), "s2_swir_assets"
    )
else:
    print("... s2_swir_assets exists; skipping Sentinel-2 download (FORCE_REBUILD=False)")

## 3. SWIR methane-proxy index + H3 grid

Read each SWIR band as a `tile` (`gtiff_gbx`), pair B12 with B11, and
`rst_mapalgebra` computes `(B11 − B12)/(B11 + B12)` (band 1 of each tile binds to
A=B12, B=B11). `rst_h3_tessellate` shreds the index raster into H3 cells → `s2_plume_cells`.

In [ ]:
def _band_tile(band):
    return (
        spark.read.format("gtiff_gbx")
        .option("filterRegex", rf".*{band}.*\.tif$")
        .load(S2_DIR)
        .select(F.col("tile").alias(f"tile_{band.lower()}"))
    )

b12 = _band_tile("B12")
b11 = _band_tile("B11")
mbmp = (
    b12.crossJoin(b11)
    .withColumn("tile", rx.rst_mapalgebra(F.array("tile_b12", "tile_b11"), "(B - A) / (A + B)"))
    .select("tile")
)
# rst_h3_tessellate is a UDTF — invoke it via SQL LATERAL (yields cellid/raster/
# metadata per overlapping H3 cell); rebuild the tile struct for rst_summary.
mbmp.createOrReplaceTempView("_mbmp")
plume_cells = spark.sql(
    f"""
    SELECT
        t.cellid AS h3_cellid,
        gbx_rst_summary(named_struct('cellid', t.cellid, 'raster', t.raster,
                                     'metadata', t.metadata)) AS stats
    FROM _mbmp s, LATERAL gbx_rst_h3_tessellate(s.tile, {S2_H3_RES}) t
    """
)
finalize_delta(plume_cells, "s2_plume_cells", do_display=False)
_pc = spark.table("s2_plume_cells")
print(f"s2_plume_cells: {_pc.count():,} H3 cells")
_pc.limit(5).display()  # limit for GitHub ipynb rendering (no geom column)

## 4. 20 m SWIR methane-proxy over the hotspot

`vizx.plot_tile` drapes the single-band MBMP index over a CartoDB basemap, masked to
the strongest-absorption pixels (top ~15%) with a **labeled colorbar** so the signal is
legible in place: **brighter = higher `(B11−B12)/(B11+B12)` = stronger relative B12
absorption = a stronger candidate methane signal** for NB03's EMIT confirmation. (Same
helper standardizes NB03's enhancement view; `plot_raster` is for RGB/multiband composites.)

In [ ]:
mbmp_tile = mbmp.first()["tile"]
show_tile(
    mbmp_tile,
    band=1,
    mask_below_percentile=85,  # show the strongest-absorption pixels over the basemap
    stretch=(2, 98),
    colorbar_label="(B11 − B12) / (B11 + B12)   —   higher = stronger SWIR absorption (candidate CH4)",
    title="Sentinel-2 SWIR methane proxy over the hotspot",
)

## What we built

- **`s2_swir_assets`** (Delta) — the staged B11/B12 SWIR COGs (Volume `band_path`).
- **`s2_plume_cells`** (Delta) — the per-H3-cell SWIR methane-proxy index.
- A **20 m SWIR proxy raster** over the flagged hotspot.

GeoBrix: `StacClient`, `gtiff_gbx`, `rst_mapalgebra`, `rst_h3_tessellate`, `rst_summary`, `gbx.vizx.plot_raster`.

Next: **notebook 03** confirms + quantifies the plume with EMIT 60 m CH4 (integrated mass enhancement + emission rate).